In [8]:
from google.colab import files
uploaded = files.upload()

Saving CW1.csv to CW1 (1).csv


In [9]:
# Installing packages

# !pip install scikit-learn pandas


# Importing libraries

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# Loading dataset

df = pd.read_csv('CW1.csv')
df.columns = df.columns.str.strip()
df = df[['Sentence', 'Resolved pattern']]

# Remove the classes with only 1 sample (clean the labels and remove rows where the label was blank or not useful)
class_counts = df['Resolved pattern'].value_counts()
df = df[df['Resolved pattern'].isin(class_counts[class_counts > 1].index)]

# Encode labels (used LabelEncoder to convert each text class into a numeric ID)
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['Resolved pattern'])


# Train-test split (spliting the data into training and testing sets using an 80–20 ratio)

X_train, X_test, y_train, y_test = train_test_split(
    df['Sentence'].values,
    df['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label'].values
)


# TF-IDF Vectorization (I converted text sentences into TF-IDF vectors.
# TF-IDF helps capture important words while reducing the weight of common words like ‘the’ or ‘is’)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


#  Logistic Regression Classifier

clf = LogisticRegression(max_iter=500)
clf.fit(X_train_tfidf, y_train)


#  Predictions & Evaluation

y_pred = clf.predict(X_test_tfidf)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_))


Test Accuracy: 0.7793103448275862

Classification Report:
                      precision    recall  f1-score   support

     ADDED COMPOUND       0.75      0.69      0.72       182
         FUSED HEAD       0.74      0.81      0.77       182
 IMPLICIT REFERENCE       0.69      0.79      0.74       180
METONYMIC REFERENCE       0.97      0.83      0.89       181

           accuracy                           0.78       725
          macro avg       0.79      0.78      0.78       725
       weighted avg       0.79      0.78      0.78       725



In [10]:
#  Train Linear SVM
from sklearn.svm import LinearSVC

svm_clf = LinearSVC()
svm_clf.fit(X_train_tfidf, y_train)

#  Predictions and Evaluation

y_pred = svm_clf.predict(X_test_tfidf)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_))


Test Accuracy: 0.7944827586206896

Classification Report:
                      precision    recall  f1-score   support

     ADDED COMPOUND       0.76      0.69      0.72       182
         FUSED HEAD       0.75      0.81      0.78       182
 IMPLICIT REFERENCE       0.73      0.82      0.77       180
METONYMIC REFERENCE       0.96      0.86      0.91       181

           accuracy                           0.79       725
          macro avg       0.80      0.79      0.80       725
       weighted avg       0.80      0.79      0.80       725



In [11]:
from xgboost import XGBClassifier

# Initialize XGBoost
xgb_clf = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

# Train the model
xgb_clf.fit(X_train_tfidf, y_train)

# Make predictions
y_pred_xgb = xgb_clf.predict(X_test_tfidf)

# Evaluation
print("XGBoost Test Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb, target_names=label_encoder.classes_))


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:53:19] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Test Accuracy: 0.7889655172413793

Classification Report:
                      precision    recall  f1-score   support

     ADDED COMPOUND       0.67      0.76      0.71       182
         FUSED HEAD       0.81      0.76      0.79       182
 IMPLICIT REFERENCE       0.78      0.77      0.77       180
METONYMIC REFERENCE       0.94      0.86      0.90       181

           accuracy                           0.79       725
          macro avg       0.80      0.79      0.79       725
       weighted avg       0.80      0.79      0.79       725



In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize Random Forest
rf_clf = RandomForestClassifier(n_estimators=200, random_state=42)

# Train the model
rf_clf.fit(X_train_tfidf, y_train)

# Make predictions
y_pred_rf = rf_clf.predict(X_test_tfidf)

# Evaluation
print("Random Forest Test Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf, target_names=label_encoder.classes_))


Random Forest Test Accuracy: 0.7655172413793103

Classification Report:
                      precision    recall  f1-score   support

     ADDED COMPOUND       0.69      0.69      0.69       182
         FUSED HEAD       0.74      0.76      0.75       182
 IMPLICIT REFERENCE       0.71      0.77      0.74       180
METONYMIC REFERENCE       0.96      0.84      0.89       181

           accuracy                           0.77       725
          macro avg       0.77      0.77      0.77       725
       weighted avg       0.77      0.77      0.77       725

